<h1 align='center'> 영상처리 프로그래밍 실습 12</h1>

<h6 align='right'> 2025. 6. 12. </h6>

<div class="alert alert-block alert-info">
    
- 파일 이름에서 00000000을 자신의 학번으로, name을 자신의 이름으로 수정하세요.

- 다음 줄에 자신의 이름, 학번, 학과(전공)을 적으세요.

* 이름:   이한석 &nbsp;&nbsp;          학번:    20215226 &nbsp;&nbsp;         학과(전공): 빅데이터학과
    
</div>

- JupyterLab 문서의 최신 버전은 [JupyterLab Documentation](https://jupyterlab.readthedocs.io/en/stable/index.html#/)을  참고하라

- Markdown은 [Markdown Guide](https://www.markdownguide.org/)를 참고하라.
- [Markdown Cheat Sheet](https://www.markdownguide.org/cheat-sheet/)


* 제출 마감: 6월 18일 (월) 오후 10:00까지 최종본을 SmartLEAD제출



### 문제 1.

MNIST handwritten digits dataset를 이용해서 직접 쓴 숫자를 인식하는 프로그램을 만들려고 한다.

### 1.1

아래 URL에서 mnist_784.arff 파일을 다운로드하라.

https://www.openml.org/search?type=data&sort=runs&id=554&status=active

(Smartlead에서도 다운로드할 수 있다.)


#### 1.2

다음 셀에 있는 프로그램을 실행하라.

In [1]:
import cv2
import pandas as pd
from scipy.io import arff
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

#### 1.3: 마우스로 숫자 쓰기 인터페이스 만들기

- 목표: OpenCV 마우스 이벤트를 활용하여 마우스로 숫자를 그릴 수 있는 인터페이스를 만든다.

- 세부 조건:
  - 검정 배경 창을 띄우고, 마우스를 움직일 때 궤적을 흰색으로 그려야 한다.
  - 왼쪽 마우스 버튼을 누른 상태에서만 선이 그려지도록 한다.
  - 창의 크기는 MNIST의 크기(28×28)의 정수배로 설정할 것 (예: 280×280)
  - **키보드 'c'**를 누르면 화면이 지워진다.
  - **키보드 'q'**를 누르면 프로그램이 종료된다.

- 힌트:
  - cv2.setMouseCallback() 함수 사용
  - cv2.line() 또는 cv2.circle()로 궤적 표시
  - drawing = True/False 상태 변수 활용


In [2]:
canvas_size = 28 * 10
canvas = np.zeros((canvas_size, canvas_size), dtype=np.uint8)

drawing = False

In [ ]:
def draw(event, x, y, flags, param):
    global drawing, canvas

    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
        cv2.circle(canvas, (x, y), 8, 255, -1) 
    elif event == cv2.EVENT_MOUSEMOVE and drawing:
        cv2.circle(canvas, (x, y), 8, 255, -1)  
    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
        cv2.circle(canvas, (x, y), 8, 255, -1)

In [4]:
cv2.namedWindow('Draw Digit')
cv2.setMouseCallback('Draw Digit', draw)

while True:
    cv2.imshow('Draw Digit', canvas)
    key = cv2.waitKey(1) & 0xFF

    if key == ord('c'):  
        canvas = np.zeros((canvas_size, canvas_size), dtype=np.uint8)
    elif key == ord('q'):  
        break

cv2.destroyAllWindows()

#### 1.4: 선 굵기 비교용 시각화
- 목표: MNIST 이미지와 직접 쓴 숫자를 나란히 비교해서, 마우스 궤적의 굵기를 확인한다.
- 세부 조건:
  - MNIST 훈련 데이터에서 첫 3개 이미지를 불러온다.
  - 마우스로 쓴 이미지를 28x28로 축소(resize) 한다.
  - MNIST 이미지들과 함께 한 화면에 나란히 표시한다.
  - 선의 굵기가 어떻게 다른지 눈으로 비교해 본다.

- 힌트:
  - MNIST 이미지: sklearn.datasets.fetch_openml("mnist_784") 또는 ARFF
  - 마우스 그림은 cv2.resize()로 줄이기




In [5]:
file_path = 'C:\\Users\\larva\\OneDrive\\바탕 화면\\Image_processing\\mnist_784.arff'
data, meta = arff.loadarff(file_path)
df = pd.DataFrame(data)
df['class'] = df['class'].apply(lambda x: int(x.decode()))

In [10]:
images = df.iloc[:3, :-1].values.reshape(-1, 28, 28).astype(np.uint8)
resized_images = [cv2.resize(img, (280, 280), interpolation=cv2.INTER_NEAREST) for img in images]


canvas = np.zeros((280, 280), dtype=np.uint8)


drawing = False

# 마우스 콜백 함수
def draw(event, x, y, flags, param):
    global drawing
    if x >= 3 * 280:
        cx = x - 3 * 280  
        cy = y
        if event == cv2.EVENT_LBUTTONDOWN:
            drawing = True
        elif event == cv2.EVENT_MOUSEMOVE and drawing:
            cv2.circle(canvas, (cx, cy), 10, 255, -1)
        elif event == cv2.EVENT_LBUTTONUP:
            drawing = False
            
cv2.namedWindow("Draw Digit - Press 'q' to Quit")
cv2.setMouseCallback("Draw Digit - Press 'q' to Quit", draw)


while True:
    combined = np.hstack(resized_images + [canvas])
    color_combined = cv2.cvtColor(combined, cv2.COLOR_GRAY2BGR)


    font = cv2.FONT_HERSHEY_SIMPLEX
    for i in range(3):
        cv2.putText(color_combined, f"MNIST {i+1}", (i * 280 + 10, 30), font, 0.8, (0, 255, 0), 2)
    cv2.putText(color_combined, "Your Drawing", (3 * 280 + 10, 30), font, 0.8, (0, 255, 0), 2)

    cv2.imshow("Draw Digit - Press 'q' to Quit", color_combined)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('c'):
        canvas[:] = 0  

cv2.destroyAllWindows()

#### 1.5

trackbar 인터페이스를 이용해서 선의 굵기를 조정할 수 있도록 1.3에서 작성한 프로그램을 수정하라.

In [ ]:
import cv2
import numpy as np
import pandas as pd
from scipy.io import arff

try:
    file_path = 'C:\\Users\\larva\\OneDrive\\바탕 화면\\Image_processing\\mnist_784.arff'
    data, meta = arff.loadarff(file_path)
    df = pd.DataFrame(data)
    df['class'] = df['class'].apply(lambda x: int(x.decode()))
    
  
    images = df.iloc[:3, :-1].values.reshape(-1, 28, 28).astype(np.uint8)
    #파일을 찾을 수 없는 경우 빈 이미지 생설
    resized_images = [cv2.resize(img, (280, 280), interpolation=cv2.INTER_NEAREST) for img in images]

except FileNotFoundError:
    print("Error: mnist_784.arff not found. Please update the file_path variable.")
    print("Using blank images as placeholders.")
    resized_images = [np.zeros((280, 280), dtype=np.uint8) for _ in range(3)]



canvas = np.zeros((280, 280), dtype=np.uint8)
drawing = False
brush_thickness = 10 #선 굵기를 위한 변수


def draw(event, x, y, flags, param):
    """캔버스에 그리기 위한 마우스 이벤트를 처리합니다."""
    global drawing
    
    if x >= 3 * 280:
        cx = x - 3 * 280  
        cy = y
        
        if event == cv2.EVENT_LBUTTONDOWN:
            drawing = True
            cv2.circle(canvas, (cx, cy), brush_thickness, 255, -1) #클릭 시 그리기
        elif event == cv2.EVENT_MOUSEMOVE and drawing:
            cv2.circle(canvas, (cx, cy), brush_thickness, 255, -1)
        elif event == cv2.EVENT_LBUTTONUP:
            drawing = False


def on_thickness_change(val):
    """트랙바가 움직일 때마다 선 굵기를 업데이트합니다."""
    global brush_thickness
    brush_thickness = max(1, val)


WINDOW_NAME = "Draw Digit - Press 'q' to Quit, 'c' to Clear"
cv2.namedWindow(WINDOW_NAME)
cv2.setMouseCallback(WINDOW_NAME, draw)

#선 굵기 조절하는 트랙바 생성
cv2.createTrackbar('Thickness', WINDOW_NAME, brush_thickness, 50, on_thickness_change)



while True:
    combined_view = np.hstack(resized_images + [canvas])
    
    
    color_combined_view = cv2.cvtColor(combined_view, cv2.COLOR_GRAY2BGR)

    font = cv2.FONT_HERSHEY_SIMPLEX
    for i in range(len(resized_images)):
        cv2.putText(color_combined_view, f"MNIST {i+1}", (i * 280 + 10, 30), font, 0.8, (0, 255, 0), 2)
    cv2.putText(color_combined_view, "Your Drawing", (len(resized_images) * 280 + 10, 30), font, 0.8, (0, 255, 0), 2)

    
    cv2.imshow(WINDOW_NAME, color_combined_view)

   
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'): 
        break
    elif key == ord('c'): 
        canvas[:] = 0  


cv2.destroyAllWindows()


#### 1.6

1.5에서 만든 글자와 MNIST의 글자 사이에 어떤 차이가 있는지 살펴 보고 마우스로 만든 글자의 특성이 MNIST 글자와 비슷하게 조작하는 방법을 생각해 보고 프로그램을 보완하라.

In [16]:
import cv2
import numpy as np
import pandas as pd
from scipy.io import arff

try:
    file_path = 'C:\\Users\\larva\\OneDrive\\바탕 화면\\Image_processing\\mnist_784.arff'
    data, meta = arff.loadarff(file_path)
    df = pd.DataFrame(data)
    df['class'] = df['class'].apply(lambda x: int(x.decode()))
    
    images = df.iloc[:3, :-1].values.reshape(-1, 28, 28).astype(np.uint8)
    resized_images = [cv2.resize(img, (280, 280), interpolation=cv2.INTER_NEAREST) for img in images]

except FileNotFoundError:
    print("오류: mnist_784.arff 파일을 찾을 수 없습니다. file_path 변수를 업데이트해주세요.")
    print("대체용으로 빈 이미지를 사용합니다.")
    # 파일을 찾을 수 없을 경우 빈 이미지 생성
    resized_images = [np.zeros((280, 280), dtype=np.uint8) for _ in range(3)]


canvas = np.zeros((280, 280), dtype=np.uint8)
drawing = False
brush_thickness = 15 # 브러시 초기 굵기를 약간 두껍게 조정


def process_to_mnist_style(img):
    """
    사용자가 그린 이미지를 MNIST 데이터셋 스타일로 변환합니다.
    1. 약간의 블러를 적용하여 선을 부드럽게 만듭니다.
    2. 이미지를 28x28로 축소하여 MNIST 해상도에 맞춥니다.
    3. 다시 280x280으로 확대하여 픽셀화된 효과를 시각적으로 보여줍니다.
    """
    # 1. 가우시안 블러 적용
    blurred_img = cv2.GaussianBlur(img, (5, 5), 0)
    
    # 2. 28x28 크기로 축소 (INTER_AREA는 축소 시 픽셀 영역의 평균을 내어 자연스러움)
    mnist_sized_img = cv2.resize(blurred_img, (28, 28), interpolation=cv2.INTER_AREA)
    
    # 3. 화면 표시를 위해 다시 280x280으로 확대 (INTER_NEAREST는 픽셀 경계를 명확하게 보여줌)
    display_img = cv2.resize(mnist_sized_img, (280, 280), interpolation=cv2.INTER_NEAREST)
    
    return display_img



# 마우스 콜백 함수
def draw(event, x, y, flags, param):
    """캔버스에 그리기 위한 마우스 이벤트를 처리합니다."""
    global drawing
    
    # 마우스가 '사용자 그림' 캔버스 영역(네 번째 칸) 위에 있는지 확인
    if 3 * 280 <= x < 4 * 280:
        cx = x - 3 * 280  
        cy = y
        
        if event == cv2.EVENT_LBUTTONDOWN:
            drawing = True
            cv2.circle(canvas, (cx, cy), brush_thickness, 255, -1) # 클릭 시 그리기
        elif event == cv2.EVENT_MOUSEMOVE and drawing:
            cv2.circle(canvas, (cx, cy), brush_thickness, 255, -1)
        elif event == cv2.EVENT_LBUTTONUP:
            drawing = False


def on_thickness_change(val):
    """트랙바가 움직일 때마다 선 굵기를 업데이트합니다."""
    global brush_thickness
    # 굵기가 최소 1이 되도록 보장
    brush_thickness = max(1, val)


WINDOW_NAME = "MNIST 스타일 숫자 그리기 - 'q': 종료, 'c': 지우기"
cv2.namedWindow(WINDOW_NAME)
cv2.setMouseCallback(WINDOW_NAME, draw)

# 선 굵기를 조절하는 트랙바 생성
cv2.createTrackbar('굵기', WINDOW_NAME, brush_thickness, 50, on_thickness_change)



while True:
    processed_canvas = process_to_mnist_style(canvas)
    
    # MNIST 이미지, 사용자 원본 그림, 변환된 그림을 수평으로 결합
    combined_view = np.hstack(resized_images + [canvas, processed_canvas])
    
    
    color_combined_view = cv2.cvtColor(combined_view, cv2.COLOR_GRAY2BGR)

    font = cv2.FONT_HERSHEY_SIMPLEX
    labels = ["MNIST 1", "MNIST 2", "MNIST 3", "DRAW HERE", "CONVERTED IMAGE"]
    for i, label in enumerate(labels):
        cv2.putText(color_combined_view, label, (i * 280 + 10, 30), font, 0.7, (0, 255, 0), 2)

    # 최종 이미지 표시
    cv2.imshow(WINDOW_NAME, color_combined_view)


    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'): # 'q'를 눌러 종료
        break
    elif key == ord('c'): # 'c'를 눌러 캔버스 지우기
        canvas[:] = 0  


cv2.destroyAllWindows()


#### 1.7

1.6에서 찾은 방법을 고려해서 예제 1.5에서 만든 프로그램을 보완하라.

단, 글자를 다 쓴 후에 오른쪽 마우스를 클릭했을 때, 1.6에서 찾은 방법을 적용한 영상이 창에 표시되어야 한다.

In [ ]:
import cv2
import numpy as np
import pandas as pd
from scipy.io import arff
import os

file_path = 'mnist_784.arff'

resized_images = []
try:
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"'{file_path}' file not found.")

    data, meta = arff.loadarff(file_path)
    df = pd.DataFrame(data)
    df['class'] = df['class'].apply(lambda x: int(x.decode()))
    
    images = df.iloc[:3, :-1].values.reshape(-1, 28, 28).astype(np.uint8)
    
    resized_images = [cv2.resize(img, (280, 280), interpolation=cv2.INTER_NEAREST) for img in images]

except FileNotFoundError as e:
    print(f"Error: {e}")
    print("mnist_784.arff file not found. Please update the file_path variable or place the file in the correct path.")
    print("Using blank images as placeholders.")
    resized_images = [np.zeros((280, 280), dtype=np.uint8) for _ in range(3)]
except Exception as e:
    print(f"An unknown error occurred during data loading: {e}")
    print("Using blank images as placeholders.")
    resized_images = [np.zeros((280, 280), dtype=np.uint8) for _ in range(3)]


canvas = np.zeros((280, 280), dtype=np.uint8)
drawing = False
brush_thickness = 15


is_converted_view = False 

def process_to_mnist_style(img):
    blurred_img = cv2.GaussianBlur(img, (5, 5), 0)
    mnist_sized_img = cv2.resize(blurred_img, (28, 28), interpolation=cv2.INTER_AREA)
    display_img = cv2.resize(mnist_sized_img, (280, 280), interpolation=cv2.INTER_NEAREST)
    return display_img


def draw(event, x, y, flags, param):
    global drawing, canvas, is_converted_view

    drawing_canvas_start_x = len(resized_images) * 280 
    drawing_canvas_end_x = drawing_canvas_start_x + 280

    if is_converted_view and event != cv2.EVENT_RBUTTONDOWN:
        return 

    if drawing_canvas_start_x <= x < drawing_canvas_end_x:
        cx = x - drawing_canvas_start_x
        cy = y
        
        if event == cv2.EVENT_LBUTTONDOWN:
            drawing = True
            is_converted_view = False 
            cv2.circle(canvas, (cx, cy), brush_thickness, 255, -1) 
        elif event == cv2.EVENT_MOUSEMOVE and drawing:
            cv2.circle(canvas, (cx, cy), brush_thickness, 255, -1)
        elif event == cv2.EVENT_LBUTTONUP:
            drawing = False
        elif event == cv2.EVENT_RBUTTONDOWN:
            if not is_converted_view:
                canvas[:] = process_to_mnist_style(canvas) 
                is_converted_view = True
                print("Right-click detected! Canvas converted to MNIST style.")
            else: 
                canvas[:] = 0 
                is_converted_view = False
                print("Right-click detected! Canvas cleared for new drawing.")


def on_thickness_change(val):
    global brush_thickness
    brush_thickness = max(1, val)

WINDOW_NAME = "MNIST Style Digit Drawing - 'q': Quit, 'c': Clear, Right-click to Convert/Reset"
cv2.namedWindow(WINDOW_NAME)
cv2.setMouseCallback(WINDOW_NAME, draw)

cv2.createTrackbar('Thickness', WINDOW_NAME, brush_thickness, 50, on_thickness_change)

while True:
    combined_view = np.hstack(resized_images + [canvas])
    
    color_combined_view = cv2.cvtColor(combined_view, cv2.COLOR_GRAY2BGR)

    font = cv2.FONT_HERSHEY_SIMPLEX

    labels = ["MNIST 1", "MNIST 2", "MNIST 3", "DRAW HERE (R-Click Convert/Reset)"]
    for i, label in enumerate(labels):
        cv2.putText(color_combined_view, label, (i * 280 + 10, 30), font, 0.7, (0, 255, 0), 2)

    cv2.imshow(WINDOW_NAME, color_combined_view)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):    
        break
    elif key == ord('c'):    
        canvas[:] = 0 
        is_converted_view = False 

cv2.destroyAllWindows()


Right-click detected! Canvas converted to MNIST style.


#### 1.8

1.7의 프로그램에 kNN 분류기를 추가해서 마우스로 쓴 글자를 인식하는 프로그램을 작성하라.

단, 오른쪽 마우스 버튼을 클릭하면 print 함수를 이용해서 인식 결과를 콘솔에 출력하라.

In [22]:
import cv2
import numpy as np
import pandas as pd
from scipy.io import arff
from sklearn.neighbors import KNeighborsClassifier
import os

file_path = 'mnist_784.arff'

resized_images = []
knn = None

try:
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"'{file_path}' file not found.")

    data, meta = arff.loadarff(file_path)
    df = pd.DataFrame(data)
    df['class'] = df['class'].apply(lambda x: int(x.decode()))

    # 이미지 배열 만들기
    images = df.iloc[:3, :-1].values.reshape(-1, 28, 28).astype(np.uint8)
    resized_images = [cv2.resize(img, (280, 280), interpolation=cv2.INTER_NEAREST) for img in images]

    # kNN 훈련 데이터 준비
    X = df.iloc[:5000, :-1].values.astype(np.uint8)  # feature
    y = df.iloc[:5000, -1].values.astype(np.uint8)   # label

    knn = KNeighborsClassifier(n_neighbors=3)
    knn.fit(X, y)

except FileNotFoundError as e:
    print(f"Error: {e}")
    print("mnist_784.arff file not found. Please update the file_path variable or place the file in the correct path.")
    print("Using blank images as placeholders.")
    resized_images = [np.zeros((280, 280), dtype=np.uint8) for _ in range(3)]
except Exception as e:
    print(f"An unknown error occurred during data loading: {e}")
    print("Using blank images as placeholders.")
    resized_images = [np.zeros((280, 280), dtype=np.uint8) for _ in range(3)]

canvas = np.zeros((280, 280), dtype=np.uint8)
drawing = False
brush_thickness = 15
is_converted_view = False

def process_to_mnist_style(img):
    blurred_img = cv2.GaussianBlur(img, (5, 5), 0)
    mnist_sized_img = cv2.resize(blurred_img, (28, 28), interpolation=cv2.INTER_AREA)
    return mnist_sized_img

def draw(event, x, y, flags, param):
    global drawing, canvas, is_converted_view

    drawing_canvas_start_x = len(resized_images) * 280 
    drawing_canvas_end_x = drawing_canvas_start_x + 280

    if is_converted_view and event != cv2.EVENT_RBUTTONDOWN:
        return

    if drawing_canvas_start_x <= x < drawing_canvas_end_x:
        cx = x - drawing_canvas_start_x
        cy = y
        
        if event == cv2.EVENT_LBUTTONDOWN:
            drawing = True
            is_converted_view = False
            cv2.circle(canvas, (cx, cy), brush_thickness, 255, -1) 
        elif event == cv2.EVENT_MOUSEMOVE and drawing:
            cv2.circle(canvas, (cx, cy), brush_thickness, 255, -1)
        elif event == cv2.EVENT_LBUTTONUP:
            drawing = False
        elif event == cv2.EVENT_RBUTTONDOWN:
            if not is_converted_view:
                mnist_img = process_to_mnist_style(canvas)
                canvas[:] = cv2.resize(mnist_img, (280, 280), interpolation=cv2.INTER_NEAREST)
                is_converted_view = True
                print("Right-click detected! Canvas converted to MNIST style.")

                # kNN 분류기 예측
                if knn:
                    input_vector = mnist_img.flatten().reshape(1, -1)
                    prediction = knn.predict(input_vector)
                    print(f"Predicted Digit: {prediction[0]}")
                else:
                    print("kNN classifier is not available.")
            else:
                canvas[:] = 0
                is_converted_view = False
                print("Right-click detected! Canvas cleared for new drawing.")

def on_thickness_change(val):
    global brush_thickness
    brush_thickness = max(1, val)

WINDOW_NAME = "MNIST Style Digit Drawing - 'q': Quit, 'c': Clear, Right-click to Convert/Recognize"
cv2.namedWindow(WINDOW_NAME)
cv2.setMouseCallback(WINDOW_NAME, draw)
cv2.createTrackbar('Thickness', WINDOW_NAME, brush_thickness, 50, on_thickness_change)

while True:
    combined_view = np.hstack(resized_images + [canvas])
    color_combined_view = cv2.cvtColor(combined_view, cv2.COLOR_GRAY2BGR)

    font = cv2.FONT_HERSHEY_SIMPLEX
    labels = ["MNIST 1", "MNIST 2", "MNIST 3", "DRAW HERE (R-Click Convert/Reset)"]
    for i, label in enumerate(labels):
        cv2.putText(color_combined_view, label, (i * 280 + 10, 30), font, 0.7, (0, 255, 0), 2)

    cv2.imshow(WINDOW_NAME, color_combined_view)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('c'):
        canvas[:] = 0
        is_converted_view = False

cv2.destroyAllWindows()


Right-click detected! Canvas converted to MNIST style.
Predicted Digit: 1
Right-click detected! Canvas converted to MNIST style.
Predicted Digit: 2
Right-click detected! Canvas converted to MNIST style.
Predicted Digit: 1
Right-click detected! Canvas converted to MNIST style.
Predicted Digit: 3
Right-click detected! Canvas converted to MNIST style.
Predicted Digit: 2
Right-click detected! Canvas converted to MNIST style.
Predicted Digit: 5


#### 1.9

cv2.putText 함수의 사용법을 조사해서, 예제 1.8의 프로그램을 수정하여 마우스로 쓴 글자의 인식 결과를 입력 영상 옆에 별도의 영상으로 출력하라.

In [29]:
import cv2
import numpy as np
import pandas as pd
from scipy.io import arff
from sklearn.neighbors import KNeighborsClassifier
import os

file_path = 'mnist_784.arff'

resized_images = [] #MNIST 이미지를 저장
knn = None #KNN 분류기 객체
canvas = np.zeros((280, 280), dtype=np.uint8) #사용자의 그림이 그려질 캔버스
prediction_display_image = np.zeros((280, 280), dtype=np.uint8) #예측된 숫자를 표시할 이미지
drawing = False
brush_thickness = 15
is_converted_view = False
drawing_active = False

try:
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"'{file_path}' file not found.")

    data, meta = arff.loadarff(file_path)
    df = pd.DataFrame(data)

    df['class'] = df['class'].apply(lambda x: int(x.decode()))

    #예시 이미지 3개
    images = df.iloc[:3, :-1].values.reshape(-1, 28, 28).astype(np.uint8)
    resized_images = [cv2.resize(img, (280, 280), interpolation=cv2.INTER_NEAREST) for img in images]

    X = df.iloc[:5000, :-1].values.astype(np.uint8)
    y = df.iloc[:5000, -1].values.astype(np.uint8)

    knn = KNeighborsClassifier(n_neighbors=3)
    knn.fit(X, y)
    print("kNN classifier trained successfully.")

except FileNotFoundError as e:
    print(f"Error: {e}")
    print("mnist_784.arff file not found. Please update the file_path variable or place the file in the correct path.")
    print("Using blank images as placeholders for example MNIST digits.")
    resized_images = [np.zeros((280, 280), dtype=np.uint8) for _ in range(3)]
except Exception as e:
    print(f"An unknown error occurred during data loading: {e}")
    print("Using blank images as placeholders for example MNIST digits.")
    resized_images = [np.zeros((280, 280), dtype=np.uint8) for _ in range(3)]

def process_to_mnist_style(img): #이미지 전처리 함수
    blurred_img = cv2.GaussianBlur(img, (5, 5), 0)
    mnist_sized_img = cv2.resize(blurred_img, (28, 28), interpolation=cv2.INTER_AREA)
    return mnist_sized_img

def draw(event, x, y, flags, param): #이밎 이벤트 콜벡 함수
    global drawing, canvas, is_converted_view, prediction_display_image, drawing_active

    drawing_canvas_start_x = len(resized_images) * 280
    drawing_canvas_end_x = drawing_canvas_start_x + 280

    if drawing_canvas_start_x <= x < drawing_canvas_end_x:
        cx = x - drawing_canvas_start_x
        cy = y

        if event == cv2.EVENT_LBUTTONDOWN:
            if drawing_active:
                drawing = True
                is_converted_view = False
                canvas[:] = 0
                prediction_display_image[:] = 0
                cv2.circle(canvas, (cx, cy), brush_thickness, 255, -1)
        elif event == cv2.EVENT_MOUSEMOVE and drawing:
            cv2.circle(canvas, (cx, cy), brush_thickness, 255, -1)
        elif event == cv2.EVENT_LBUTTONUP:
            drawing = False
        elif event == cv2.EVENT_RBUTTONDOWN: 
            if not is_converted_view:
                if not drawing_active:
                    drawing_active = True
                    print("오른쪽 클릭 감지! 그리기 모드가 활성화되었습니다.")
                else:
                    mnist_img = process_to_mnist_style(canvas)
                    canvas[:] = cv2.resize(mnist_img, (280, 280), interpolation=cv2.INTER_NEAREST)
                    is_converted_view = True
                    drawing_active = False
                    print("오른쪽 클릭 감지! 캔버스가 MNIST 스타일로 변환되었습니다.")

                    if knn:
                        input_vector = mnist_img.flatten().reshape(1, -1)
                        prediction = knn.predict(input_vector)
                        predicted_digit = prediction[0]
                        print(f"Predicted Digit: {predicted_digit}")

                        prediction_display_image[:] = 0
                        text = str(predicted_digit)
                        font = cv2.FONT_HERSHEY_SIMPLEX
                        font_scale = 10
                        font_thickness = 20
                        text_size = cv2.getTextSize(text, font, font_scale, font_thickness)[0]
                        text_x = (prediction_display_image.shape[1] - text_size[0]) // 2
                        text_y = (prediction_display_image.shape[0] + text_size[1]) // 2
                        cv2.putText(prediction_display_image, text, (text_x, text_y), font, font_scale, 255, font_thickness, cv2.LINE_AA)
                    else:
                        print("kNN classifier is not available. Cannot predict.")
                        prediction_display_image[:] = 0
            else:
                canvas[:] = 0
                is_converted_view = False
                drawing_active = True
                prediction_display_image[:] = 0
                print("Right-click detected! Canvas cleared for new drawing. Drawing mode activated.")

def on_thickness_change(val): #트랙바 콜백 함수
    global brush_thickness
    brush_thickness = max(1, val)

WINDOW_NAME = "MNIST Style Digit Drawing - 'q': Quit, 'c': Clear, Right-click to Convert/Recognize"
cv2.namedWindow(WINDOW_NAME)
cv2.setMouseCallback(WINDOW_NAME, draw)
cv2.createTrackbar('Thickness', WINDOW_NAME, brush_thickness, 50, on_thickness_change)


#DRAW HERE에 오른쪽 마우스 클릭하면 그리기 가능
#그린 후 오른쪽 마우스 클릭하면 MNIST 스타일로 변환
while True:
    combined_view = np.hstack(resized_images + [canvas, prediction_display_image])
    color_combined_view = cv2.cvtColor(combined_view, cv2.COLOR_GRAY2BGR)

    font = cv2.FONT_HERSHEY_SIMPLEX
    labels = ["MNIST 1", "MNIST 2", "MNIST 3", "DRAW HERE", "PREDICTED DIGIT"]
    for i, label in enumerate(labels):
        cv2.putText(color_combined_view, label, (i * 280 + 10, 30), font, 0.7, (0, 255, 0), 2)

    cv2.imshow(WINDOW_NAME, color_combined_view)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('c'):
        canvas[:] = 0
        is_converted_view = False
        drawing_active = False
        prediction_display_image[:] = 0

cv2.destroyAllWindows()


kNN classifier trained successfully.
오른쪽 클릭 감지! 그리기 모드가 활성화되었습니다.
오른쪽 클릭 감지! 캔버스가 MNIST 스타일로 변환되었습니다.
Predicted Digit: 1
Right-click detected! Canvas cleared for new drawing. Drawing mode activated.
오른쪽 클릭 감지! 캔버스가 MNIST 스타일로 변환되었습니다.
Predicted Digit: 2
오른쪽 클릭 감지! 그리기 모드가 활성화되었습니다.
오른쪽 클릭 감지! 그리기 모드가 활성화되었습니다.
오른쪽 클릭 감지! 캔버스가 MNIST 스타일로 변환되었습니다.
Predicted Digit: 3
오른쪽 클릭 감지! 그리기 모드가 활성화되었습니다.
오른쪽 클릭 감지! 캔버스가 MNIST 스타일로 변환되었습니다.
Predicted Digit: 3
오른쪽 클릭 감지! 그리기 모드가 활성화되었습니다.
오른쪽 클릭 감지! 캔버스가 MNIST 스타일로 변환되었습니다.
Predicted Digit: 0
오른쪽 클릭 감지! 그리기 모드가 활성화되었습니다.
오른쪽 클릭 감지! 캔버스가 MNIST 스타일로 변환되었습니다.
Predicted Digit: 1
